# EDA 03 — Base tratada y reserva de prueba
**Objetivo:** comprender y verificar las transformaciones antes de explorar asociaciones o entrenar modelos.

El programa `src/02_preparar_base.py` genera los productos y `src/02b_verificar_bases.py` los contrasta con el original. Este cuaderno lee sus resultados: no vuelve a dividir la muestra ni entrena modelos. Ejecuta las celdas en orden. Los resultados son registros de encuesta, no personas únicas seguidas longitudinalmente.

## 1. Localizar los archivos
Una ruta indica dónde está un archivo; una tabla de pandas permite trabajar con sus filas y columnas.

In [1]:
# ETAPA 3 / ARCHIVO: cuaderno 03 / BLOQUE 1: ubicación y dependencias.
from pathlib import Path  # Representa carpetas y archivos.
import json  # Lee los informes estructurados.
import importlib.util  # Carga el lector del proyecto.
import pandas as pd  # Trabaja con tablas.
from IPython.display import display  # Presenta las tablas legibles.
RAIZ = Path.cwd() if (Path.cwd() / 'data/processed').exists() else Path.cwd().parent  # Admite iniciar desde proyecto o notebooks.
assert (RAIZ / 'src/02a_leer_bases.py').exists(), 'Abre el cuaderno desde notebooks o desde el proyecto.'  # Verifica la ubicación.
REPORTES = RAIZ / 'reports/preparacion'  # Localiza la evidencia.
resumen = json.loads((REPORTES / '00_resumen_preparacion.json').read_text(encoding='utf-8'))  # Lee el resumen.
verificacion = json.loads((REPORTES / '10_verificacion_bases.json').read_text(encoding='utf-8'))  # Lee la comprobación independiente.
assert verificacion['estado'] == 'verificado'  # Exige que la revisión haya terminado.
display(pd.DataFrame(resumen['productos'])[['archivo', 'filas', 'columnas']])  # Muestra el inventario sin identificadores personales.

,archivo,filas,columnas
0,personas_tratada_completa.csv.gz,334786,165
1,dominio_educacion_superior.csv,39551,165
2,ocupados_ingresos.csv,36906,18
3,modelado/entrenamiento.csv,31502,20
4,modelado/prueba_reservada.csv,7791,20


## 2. Comprender la selección
La base completa conserva 334 786 registros y las 139 variables originales, normalizando espacios y el formato numérico del peso. Se agregan 26 variables. El dominio exige edad ≥15, pertenecer a la PEA, nivel superior reportado y título (`p12a=1`). Las 258 observaciones con `condact=6` permanecen en el dominio descriptivo, pero no tienen etiqueta para el modelo principal.

**Ejercicio:** explica por qué 39 551 y 39 293 son tamaños de conjuntos diferentes y ambos son correctos.

In [2]:
# BLOQUE 2: flujo de selección y trazabilidad.
flujo = pd.read_csv(REPORTES / '02_flujo_seleccion.csv')  # Recupera cada filtro y su recuento.
display(flujo)  # Presenta la selección acumulada.
bitacora = pd.read_csv(REPORTES / '01_bitacora_transformaciones.csv')  # Recupera los cambios de formato.
display(bitacora.head(10))  # Ilustra el registro por variable.
assert resumen['filas_eliminadas_muestra_completa'] == 0  # Comprueba que la base completa no perdió registros.
assert resumen['celdas_imputadas'] == 0  # Comprueba que no se inventaron valores.

,etapa,registros_antes,registros_despues,excluidos_en_paso
0,Original completo,334786,334786,0
1,Edad conocida y al menos 15,334786,266889,67897
2,Dentro de la PEA,266889,169445,97444
3,Nivel superior reportado,169445,50863,118582
4,Título superior declarado,50863,39551,11312
5,Etiqueta laboral principal disponible,39551,39293,258


,variable,operacion,celdas_con_espacios_retirados,celdas_ausentes,filas_eliminadas
0,area,espacios_exteriores_y_vacio_a_ausente,0,0,0
1,ciudad,espacios_exteriores_y_vacio_a_ausente,0,0,0
2,conglomerado,espacios_exteriores_y_vacio_a_ausente,0,0,0
3,panelm,espacios_exteriores_y_vacio_a_ausente,0,0,0
4,vivienda,espacios_exteriores_y_vacio_a_ausente,0,0,0
5,hogar,espacios_exteriores_y_vacio_a_ausente,0,0,0
6,p01,espacios_exteriores_y_vacio_a_ausente,0,0,0
7,p02,espacios_exteriores_y_vacio_a_ausente,0,0,0
8,p03,espacios_exteriores_y_vacio_a_ausente,0,0,0
9,p04,espacios_exteriores_y_vacio_a_ausente,0,0,0


## 3. Leer con tipos explícitos y separar usos
Los identificadores son texto: sus ceros iniciales tienen significado. El lector utiliza el esquema guardado. `X` contiene solamente las diez características permitidas; `y`, el resultado; y `contexto`, los pesos y grupos. No debe entrenarse con todas las columnas de la base completa.

El código 98 de edad significa **98 años o más**: se conserva como límite inferior y se acompaña de un indicador. Los ingresos no son predictores. Los códigos −1 y 999999 se conservan en el campo original y se excluyen del nuevo monto utilizable; el cero real se conserva. No se imputaron ni recortaron ingresos.

In [3]:
# BLOQUE 3: cargar únicamente entrenamiento con el lector seguro.
spec = importlib.util.spec_from_file_location('lector', RAIZ / 'src/02a_leer_bases.py')  # Localiza el módulo de lectura.
lector = importlib.util.module_from_spec(spec)  # Prepara el módulo.
spec.loader.exec_module(lector)  # Carga sus funciones.
X, y, contexto = lector.cargar_modelado()  # Lee entrenamiento; mantiene prueba reservada.
assert len(X) == 31502 and X.shape[1] == 10  # Verifica tamaño y lista de características.
assert list(X.columns) == resumen['predictores']  # Exige la lista preespecificada.
display(X.dtypes.astype(str).rename('tipo').to_frame())  # Muestra tipos sin publicar registros individuales.
diccionario = pd.read_csv(RAIZ / 'data/processed/diccionario_base_tratada.csv')  # Abre las definiciones.
display(diccionario.tail(26))  # Explica todas las variables derivadas.

,tipo
edad_limite_inferior,Int16
edad_98_mas,Int8
p02,string
p06,string
p07,string
p10a,string
p15,string
prov,string
area,string
mes,string


,variable,origen,descripcion,tipo,rol,regla,ausente_significa
139,edad_limite_inferior,p03,Edad en años; 98 es límite inferior de 98 y más,Int16,predictor_numerico,0..97 edad exacta; 98 límite inferior; 99 o va...,No aplicable o desconocido según dominio
140,edad_98_mas,p03,Indicador de categoría de edad abierta,Int8,predictor_numerico,1 si p03=98; 0 si 0..97; ausente si edad desco...,No aplicable o desconocido según dominio
141,motivo_elegibilidad,"p03,condact,p10a,p12a",Primera razón de exclusión o inclusión,string,trazabilidad,"Edad conocida >=15; PEA; nivel 8..10; p12a=1, ...",Nunca ausente
142,dominio_estudio,motivo_elegibilidad,PEA con título superior declarado y edad compa...,boolean,dominio_inferencia,True si motivo_elegibilidad=incluido,Nunca ausente
143,dominio_ingresos,"dominio_estudio,condact",Ocupados del dominio educativo,boolean,dominio_ingresos,dominio_estudio y condact en 1..6,Nunca ausente
144,adecuado_descriptivo,"condact,dominio_estudio",Pertenencia a categoría oficial de empleo adec...,Int8,resultado_descriptivo,1 si condact=1; 0 si 2..8 dentro del dominio,Fuera del dominio del estudio
145,en_modelo_principal,"dominio_estudio,condact",Registros con etiqueta principal utilizable,boolean,dominio_predictivo,dominio_estudio excluyendo condact=6,Nunca ausente
146,y_adecuado,"adecuado_descriptivo,en_modelo_principal",Etiqueta binaria para clasificación principal,Int8,resultado_predictivo,"1 adecuado; 0 condact 2..5,7..8; ausente conda...",No aplicable o desconocido según dominio
147,subempleo_descriptivo,"condact,dominio_estudio",subempleo descriptivo,Int8,resultado_descriptivo,"1 si condact en ['2', '3']; 0 otras categorías...",No aplicable o desconocido según dominio
148,desempleo_descriptivo,"condact,dominio_estudio",desempleo descriptivo,Int8,resultado_descriptivo,"1 si condact en ['7', '8']; 0 otras categorías...",No aplicable o desconocido según dominio


## 4. Verificar la reserva por UPM
Se reservaron aproximadamente 20 % de las UPM **dentro de cada estrato**, con semilla 20250928 y ordenamiento mediante SHA-256. El porcentaje de registros no necesita ser exactamente 20 %. La asignación no utiliza el resultado laboral. Las UPM de entrenamiento tienen cinco pliegues internos para la futura validación.

Las claves sin mes permiten comprobar las revisitas. Cero solapamientos de claves no demuestra que una persona que cambió de vivienda no pueda aparecer con otro identificador. El estrato 1321 no tiene casos del dominio en prueba; se documenta sin cambiar la semilla. No se evaluaron modelos ni métricas en prueba.

In [4]:
# BLOQUE 4: tamaños y comprobación de separación.
display(pd.DataFrame(resumen['particiones']))  # Muestra tamaños por uso, sin tasas del resultado.
display(pd.DataFrame(resumen['solapamientos']))  # Muestra intersecciones de claves.
assert verificacion['solapamientos_entre_particiones'] == 0  # Exige independencia de los grupos entre conjuntos.
assert verificacion['solapamientos_entre_pliegues'] == 0  # Exige separación de grupos en validación interna.
assert verificacion['prueba_evaluada'] is False  # Comprueba el estado declarado de reserva.
display(pd.read_csv(REPORTES / '06_pliegues_entrenamiento.csv'))  # Presenta los cinco pliegues.
print('Estratos del dominio sin casos en prueba:', verificacion['estratos_dominio_sin_casos_en_prueba'])  # Hace visible la limitación.

,uso,particion,registros,upm,estratos,suma_fexp_original
0,muestra_completa,entrenamiento,267761,6202,150,1.536748e+07
1,muestra_completa,prueba,67025,1578,150,3.487561e+06
2,dominio_estudio,entrenamiento,31722,5057,150,1.114762e+06
3,dominio_estudio,prueba,7829,1269,149,2.680123e+05
4,modelo_principal,entrenamiento,31502,5054,150,1.105198e+06
5,modelo_principal,prueba,7791,1269,149,2.670041e+05


,clave,claves_entrenamiento,claves_prueba,claves_compartidas
0,upm,6202,1578,0
1,id_vivienda_sin_mes,53642,13475,0
2,id_hogar_sin_mes,53868,13540,0
3,clave_posicion_sin_mes,175984,43794,0


,pliegue_cv,registros,upm
0,1,6478,1057
1,2,6494,1045
2,3,6451,1011
3,4,6267,1013
4,5,5812,928


Estratos del dominio sin casos en prueba: ['1321']


## 5. Cierre y reproducción
Se contrastaron las 139 columnas originales en las 334 786 filas, se verificaron las huellas de los cinco productos y se reprodujo independientemente la asignación de UPM. El balance de covariables se examinó una vez; no se ajustó la semilla para mejorarlo. La mayor diferencia absoluta observada entre las proporciones categóricas comparadas fue 1,444 puntos porcentuales; esto no garantiza representatividad de cada subgrupo.

Para reconstruir, ejecutar en orden `src/02_preparar_base.py` y `src/02b_verificar_bases.py`, y luego este cuaderno. La configuración está en `config/preparacion_v1.json`. Los CSV tratados usan coma como separador y punto decimal; el lector conserva los tipos. La base completa y las variables del diseño deben mantenerse para la futura inferencia de dominios; el CSV filtrado no sustituye esa especificación.

**Siguiente etapa:** análisis univariado, con decisiones de modelado basadas en entrenamiento. Los descriptivos anuales y la evaluación predictiva se documentarán por separado. Ver `docs/06_informe_preparacion.md` para fuentes y decisiones.

**Ejercicio final:** identifica qué archivo usarías para describir la población, desarrollar un modelo y evaluarlo al final. Explica por qué no usarías ingresos para predecir empleo adecuado.